In [10]:
import csv
import random

columns = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin', 'car_name']

# ==========================================
# TASK 1: EXPLORATION & PREPROCESSING
# ==========================================

# 1. Read dataset
data = []
with open('auto_mpg.csv', 'r') as file:
    reader = csv.reader(file)
    header = next(reader)  # Skip header line
    for row in reader:
        if row:
            data.append(row)

print("Shape:", (len(data), len(columns)))

# 2. Find missing values and collect valid horsepower numbers
hp_index = columns.index('horsepower')
valid_hp = []
missing_count = 0

for row in data:
    val = row[hp_index].strip()
    if val == '?':
        missing_count += 1
    else:
        valid_hp.append(float(val))

# 3. Calculate median horsepower
valid_hp.sort()
n = len(valid_hp)
if n % 2 == 1:
    median_hp = valid_hp[n // 2]
else:
    median_hp = (valid_hp[n // 2 - 1] + valid_hp[n // 2]) / 2.0

print("Missing '?' count:", missing_count)
print("Horsepower Median:", median_hp)

# 4. Replace '?' with median & Convert Data Types
clean_data = []
for row in data:
    hp = float(row[hp_index].strip()) if row[hp_index].strip() != '?' else median_hp

    cleaned_row = [
        float(row[0]),  # mpg
        int(row[1]),    # cylinders
        float(row[2]),  # displacement
        hp,             # horsepower
        float(row[4]),  # weight
        float(row[5]),  # acceleration
        int(row[6]),    # model_year
        int(row[7]),    # origin
        row[8].strip().strip('"\'')  # car_name
    ]
    clean_data.append(cleaned_row)

# 5. Remove Duplicate Rows
unique_data = []
for row in clean_data:
    if row not in unique_data:
        unique_data.append(row)

print("Original Rows:", len(data))
print("Rows after removing duplicates:", len(unique_data))
print("\nFirst cleaned row sample:")
print(unique_data[0])

# 6. Basic Analysis & Aggregations
avg_mpg = sum(row[0] for row in unique_data) / len(unique_data)
avg_weight = sum(row[4] for row in unique_data) / len(unique_data)
print(f"\nAverage MPG: {avg_mpg:.2f}")
print(f"Average Weight: {avg_weight:.2f}")

# Average MPG by Origin (1: USA, 2: Europe, 3: Japan)
origin_mpg = {1: [], 2: [], 3: []}
origin_names = {1: "USA", 2: "Europe", 3: "Japan"}

for row in unique_data:
    origin = row[7]
    mpg = row[0]
    if origin in origin_mpg:
        origin_mpg[origin].append(mpg)

print("\n--- Average MPG by Origin ---")
for origin_id, mpg_list in origin_mpg.items():
    if mpg_list:
        avg_origin_mpg = sum(mpg_list) / len(mpg_list)
        print(f"Origin {origin_id} ({origin_names[origin_id]}): {avg_origin_mpg:.2f} MPG")

# 7. Export Cleaned Data into a new file
output_file = 'cleaned_auto_mpg.csv'
with open(output_file, 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(columns)
    writer.writerows(unique_data)

print(f"\nCleaned dataset saved successfully to: {output_file}")


# ==========================================
# TASK 2: LINEAR REGRESSION
# ==========================================

# 1. Read cleaned dataset
data = []
with open('auto_mpg.csv', 'r') as file:
    reader = csv.reader(file)
    next(reader)  # Skip header row
    for row in reader:
        if row:
            data.append(row)

# 2. Extract Features (x = weight) and Target (y = mpg)
# Excluding 'car_name' as requested
x_all = [float(row[4]) for row in data]  # Weight
y_all = [float(row[0]) for row in data]  # MPG

# 3. Split into 80% Train and 20% Test (using fixed seed for reproducibility)
random.seed(42)
indices = list(range(len(x_all)))
random.shuffle(indices)

split_point = int(0.8 * len(x_all))
train_indices = indices[:split_point]
test_indices = indices[split_point:]

x_train = [x_all[i] for i in train_indices]
y_train = [y_all[i] for i in train_indices]
x_test = [x_all[i] for i in test_indices]
y_test = [y_all[i] for i in test_indices]

# 4. Calculate Linear Regression Slope (m) and Intercept (c) on TRAIN data
n_train = len(x_train)
mean_x = sum(x_train) / n_train
mean_y = sum(y_train) / n_train

num = sum((x_train[i] - mean_x) * (y_train[i] - mean_y) for i in range(n_train))
den = sum((x_train[i] - mean_x) ** 2 for i in range(n_train))

m = num / den
c = mean_y - (m * mean_x)

print(f"\nRegression Line: Predicted MPG = ({m:.6f} * Weight) + {c:.4f}\n")

# 5. Evaluate Predictions on TEST data
n_test = len(x_test)
absolute_errors = []
squared_errors = []

for i in range(n_test):
    actual_mpg = y_test[i]
    predicted_mpg = (m * x_test[i]) + c
    error = actual_mpg - predicted_mpg

    absolute_errors.append(abs(error))
    squared_errors.append(error ** 2)

# 6. Compute Error Metrics
mae = sum(absolute_errors) / n_test
mse = sum(squared_errors) / n_test
rmse = mse ** 0.5

print("--- Test Error Metrics ---")
print(f"MAE  (Mean Absolute Error): {mae:.4f}")
print(f"MSE  (Mean Squared Error):  {mse:.4f}")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")


Shape: (398, 9)
Missing '?' count: 6
Horsepower Median: 93.5
Original Rows: 398
Rows after removing duplicates: 398

First cleaned row sample:
[18.0, 8, 307.0, 130.0, 3504.0, 12.0, 70, 1, 'chevrolet chevelle malibu']

Average MPG: 23.51
Average Weight: 2970.42

--- Average MPG by Origin ---
Origin 1 (USA): 20.08 MPG
Origin 2 (Europe): 27.89 MPG
Origin 3 (Japan): 30.45 MPG

Cleaned dataset saved successfully to: cleaned_auto_mpg.csv

Regression Line: Predicted MPG = (-0.007809 * Weight) + 46.8535

--- Test Error Metrics ---
MAE  (Mean Absolute Error): 3.8411
MSE  (Mean Squared Error):  24.3484
RMSE (Root Mean Squared Error): 4.9344
